# Arbitrage-Consistent Projection directly on bid--ask option quotes

This notebook implements the **Arbitrage-Consistent Projection (ACP)** directly on bid--ask quotes for European calls and puts.

In the paper, the ACP is first written for one price per option, namely for a system

$$
(C_i,P_i)_{i=1}^n,
$$

mainly for notational simplicity. In practical market data, however, traded option quotes are usually given as bid--ask intervals:

$$
C_i^{bid} \leq C_i^{ask},
\qquad
P_i^{bid} \leq P_i^{ask},
\qquad i=1,\ldots,n.
$$

The purpose of the present code is to correct the endpoints of these intervals directly. That is, it produces

$$
C_i^{bid,ACP},\quad C_i^{ask,ACP},\quad
P_i^{bid,ACP},\quad P_i^{ask,ACP},
\qquad i=1,\ldots,n,
$$

so that the corrected bid--ask system is statically consistent with no-arbitrage.

---

## 1. Input of the algorithm

For one fixed valuation date and one fixed maturity, the input consists of:

- the current underlying price:

$$
S>0,
$$

- the strikes:

$$
0<K_1<\cdots<K_n,
$$

- the continuously compounded risk-free rate:

$$
r,
$$

- the time to maturity:

$$
\tau>0,
$$

- the bid--ask quotes of the calls:

$$
C_i^{bid},\quad C_i^{ask},
\qquad i=1,\ldots,n,
$$

- the bid--ask quotes of the puts:

$$
P_i^{bid},\quad P_i^{ask},
\qquad i=1,\ldots,n.
$$

In the code, these are represented by the arrays

```python
S
K
r
tau
call_bid
call_ask
put_bid
put_ask
```

The main ACP routine is called as follows:

```python
result = acp_direct_bid_ask(
    S=S,
    K=K,
    r=r,
    tau=tau,
    call_bid=call_bid,
    call_ask=call_ask,
    put_bid=put_bid,
    put_ask=put_ask,
)
```

---

## 2. Output of the algorithm

The ACP returns the corrected bid--ask quotes:

```python
result["call_bid_acp"]
result["call_ask_acp"]
result["put_bid_acp"]
result["put_ask_acp"]
```

That is, it returns

$$
C_i^{bid,ACP},\quad C_i^{ask,ACP},
\qquad
P_i^{bid,ACP},\quad P_i^{ask,ACP}.
$$

These are the main financial outputs of the algorithm.

The ACP chooses the corrections so that they are as small as possible in a weighted \(L^1\) sense relative to the original bid--ask quotes. More precisely, it solves a problem of the form

$$
\min
\sum_{i=1}^n
\left(
w_i^{C,bid}\left|C_i^{bid,ACP}-C_i^{bid}\right|
+
w_i^{C,ask}\left|C_i^{ask,ACP}-C_i^{ask}\right|
+
w_i^{P,bid}\left|P_i^{bid,ACP}-P_i^{bid}\right|
+
w_i^{P,ask}\left|P_i^{ask,ACP}-P_i^{ask}\right|
\right),
$$

subject to static no-arbitrage consistency constraints.

---

## 3. Bid--ask consistency constraints

First, the corrected system must satisfy bid--ask consistency:

$$
C_i^{bid,ACP} \leq C_i^{ask,ACP},
\qquad
P_i^{bid,ACP} \leq P_i^{ask,ACP},
\qquad i=1,\ldots,n.
$$

We also impose nonnegativity of the corrected quotes:

$$
C_i^{bid,ACP}\geq 0,\quad C_i^{ask,ACP}\geq 0,
$$

and

$$
P_i^{bid,ACP}\geq 0,\quad P_i^{ask,ACP}\geq 0.
$$

This automatically corrects crossed markets, such as cases where the raw data contain

$$
C_i^{bid}>C_i^{ask}.
$$

---

## 4. Internal no-arbitrage certificate

The ACP uses an internal dual certificate to enforce static no-arbitrage consistency.

Define the nodes

$$
x_0=0,
\qquad
x_j=K_j,\quad j=1,\ldots,n.
$$

The certificate consists of positive variables

$$
\lambda_0,\lambda_1,\ldots,\lambda_n,\beta>0.
$$

They satisfy

$$
\sum_{j=0}^n \lambda_j=e^{-r\tau},
$$

and

$$
\sum_{j=0}^n \lambda_j x_j+\beta=S.
$$

The quantities

$$
\widetilde C_i
=
\sum_{j=0}^n \lambda_j(x_j-K_i)^+ + \beta,
$$

and

$$
\widetilde P_i
=
\sum_{j=0}^n \lambda_j(K_i-x_j)^+
$$

play the role of internal arbitrage-free representative values.

Important: these quantities, \(\widetilde C_i\) and \(\widetilde P_i\), are not the final ACP outputs. They are only an internal certificate proving that the corrected bid--ask system contains an arbitrage-free frictionless system.

Therefore, the corrected bid--ask system is required to satisfy

$$
C_i^{bid,ACP}
\leq
\widetilde C_i
\leq
C_i^{ask,ACP},
\qquad i=1,\ldots,n,
$$

and

$$
P_i^{bid,ACP}
\leq
\widetilde P_i
\leq
P_i^{ask,ACP},
\qquad i=1,\ldots,n.
$$

Thus, the final result is a corrected bid--ask system inside which there exists a statically consistent arbitrage-free price system.

---

## 5. Why the certificate is sufficient

If there exist positive variables

$$
\lambda_0,\ldots,\lambda_n,\beta>0
$$

satisfying the relations above, then there exists a strictly positive linear pricing functional on the finite payoff space generated by:

- the bank account,
- the underlying asset,
- the traded calls,
- the traded puts.

This rules out deterministic static arbitrage for the corresponding option system.

In the bid--ask case, the logic is the following: we do not require a unique corrected option price. Instead, we require the corrected bid--ask system to be sufficiently consistent so that it contains at least one arbitrage-free frictionless price system. This is the meaning of the constraints

$$
C_i^{bid,ACP}
\leq
\widetilde C_i
\leq
C_i^{ask,ACP},
$$

and

$$
P_i^{bid,ACP}
\leq
\widetilde P_i
\leq
P_i^{ask,ACP}.
$$

---

## 6. Routines included in the code

The code contains the following main routines.

### `bs_call_put`

```python
call, put = bs_call_put(S, K, r, sigma, tau)
```

This routine computes Black--Scholes prices for European calls and puts.

It is used only for testing or simulation.

It is not needed if we already have real option data from a market source, a CSV file, an Excel file, a database, or an API.

In real-data applications, this routine can be ignored completely.

### `make_disturbed_bid_ask_test`

```python
data = make_disturbed_bid_ask_test()
```

This routine generates a synthetic option chain from Black--Scholes prices and then adds noise or artificial bid--ask/no-arbitrage violations.

It is used only to test the ACP algorithm.

It is not needed if external option data are already available.

### `acp_direct_bid_ask`

```python
result = acp_direct_bid_ask(
    S=S,
    K=K,
    r=r,
    tau=tau,
    call_bid=call_bid,
    call_ask=call_ask,
    put_bid=put_bid,
    put_ask=put_ask,
)
```

This is the main ACP routine.

It works directly on bid--ask option quotes.

If real market data are available, this is the only ACP function that needs to be called.

The function returns

```python
result["call_bid_acp"]
result["call_ask_acp"]
result["put_bid_acp"]
result["put_ask_acp"]
```

namely the corrected bid--ask option chain.

Optionally, it also returns the internal certificate:

```python
result["lambda"]
result["beta"]
result["certificate_call"]
result["certificate_put"]
```

These quantities are useful for diagnostics, but they are not the main financial output.

### `check_acp_bid_ask_result`

```python
check_acp_bid_ask_result(K, result)
```

This routine checks whether the corrected system satisfies the basic conditions

$$
C_i^{bid,ACP} \leq C_i^{ask,ACP},
$$

$$
P_i^{bid,ACP} \leq P_i^{ask,ACP},
$$

and whether the internal certificate lies inside the corrected bid--ask intervals.

It is used for diagnostics after running the ACP.

---

## 7. Using real data instead of Black--Scholes data

If option data are obtained from a CSV file, Excel file, Bloomberg, Yahoo Finance, a broker API, a database, or another external source, then the routines `bs_call_put` and `make_disturbed_bid_ask_test` are not needed.

We only need to construct the arrays

```python
S
K
r
tau
call_bid
call_ask
put_bid
put_ask
```

and then call

```python
result = acp_direct_bid_ask(
    S=S,
    K=K,
    r=r,
    tau=tau,
    call_bid=call_bid,
    call_ask=call_ask,
    put_bid=put_bid,
    put_ask=put_ask,
)
```

---

## 8. Example with CSV data

Suppose we have a file `option_chain.csv` with columns

```text
strike, call_bid, call_ask, put_bid, put_ask
```

Then we can run:

```python
df = pd.read_csv("option_chain.csv")

df = df.sort_values("strike")

K = df["strike"].to_numpy(float)
call_bid = df["call_bid"].to_numpy(float)
call_ask = df["call_ask"].to_numpy(float)
put_bid = df["put_bid"].to_numpy(float)
put_ask = df["put_ask"].to_numpy(float)

S = 100.0
r = 0.03
tau = 0.50

result = acp_direct_bid_ask(
    S=S,
    K=K,
    r=r,
    tau=tau,
    call_bid=call_bid,
    call_ask=call_ask,
    put_bid=put_bid,
    put_ask=put_ask,
)

df["call_bid_acp"] = result["call_bid_acp"]
df["call_ask_acp"] = result["call_ask_acp"]
df["put_bid_acp"] = result["put_bid_acp"]
df["put_ask_acp"] = result["put_ask_acp"]

display(df)
```

---

## 9. Multiple maturities

The ACP should be applied separately for each maturity.

If the data contain several expirations, for example through a column called `maturity`, then we can apply the ACP maturity by maturity:

```python
results = []

for maturity, g in df.groupby("maturity"):
    g = g.sort_values("strike").copy()

    K = g["strike"].to_numpy(float)
    call_bid = g["call_bid"].to_numpy(float)
    call_ask = g["call_ask"].to_numpy(float)
    put_bid = g["put_bid"].to_numpy(float)
    put_ask = g["put_ask"].to_numpy(float)

    tau = g["tau"].iloc[0]

    res = acp_direct_bid_ask(
        S=S,
        K=K,
        r=r,
        tau=tau,
        call_bid=call_bid,
        call_ask=call_ask,
        put_bid=put_bid,
        put_ask=put_ask,
    )

    g["call_bid_acp"] = res["call_bid_acp"]
    g["call_ask_acp"] = res["call_ask_acp"]
    g["put_bid_acp"] = res["put_bid_acp"]
    g["put_ask_acp"] = res["put_ask_acp"]

    results.append(g)

df_acp = pd.concat(results, ignore_index=True)
display(df_acp)
```

---

## 10. How to choose the weights

If no weights are provided, the ACP assigns the same correction cost to every bid and ask endpoint.

However, different weights may be used. For example, if a quote is considered more reliable or more liquid, then it can be assigned a larger weight so that the ACP moves it less.

A practical choice is to set the weights inversely proportional to the bid--ask spread:

$$
w_i \approx \frac{1}{\text{spread}_i}.
$$

For calls, this gives

$$
w_i^{C,bid}
=
w_i^{C,ask}
=
\frac{1}{C_i^{ask}-C_i^{bid}+\varepsilon},
$$

and for puts,

$$
w_i^{P,bid}
=
w_i^{P,ask}
=
\frac{1}{P_i^{ask}-P_i^{bid}+\varepsilon}.
$$

This means that tight spreads are moved less, while wider spreads are allowed to adjust more.

In code, this can be written as:

```python
eps = 1e-6

call_spread = np.maximum(call_ask - call_bid, eps)
put_spread = np.maximum(put_ask - put_bid, eps)

weights = {
    "call_bid": 1.0 / call_spread,
    "call_ask": 1.0 / call_spread,
    "put_bid": 1.0 / put_spread,
    "put_ask": 1.0 / put_spread,
}

result = acp_direct_bid_ask(
    S=S,
    K=K,
    r=r,
    tau=tau,
    call_bid=call_bid,
    call_ask=call_ask,
    put_bid=put_bid,
    put_ask=put_ask,
    weights=weights,
)
```

---

## 11. Interpretation of the ACP objective value

The LP returns an objective value:

```python
result["objective"]
```

If

$$
\text{objective}=0,
$$

then the original bid--ask quotes were already compatible with the ACP constraints.

If

$$
\text{objective}>0,
$$

then the ACP had to move some bid or ask quotes in order to remove static inconsistencies.

The objective value is the total weighted size of the correction.

---

## 12. Practical interpretation

The original bid--ask system

$$
\left(
C_i^{bid},
C_i^{ask},
P_i^{bid},
P_i^{ask}
\right)_{i=1}^n
$$

may contain errors, noise, crossed quotes, or violations of static no-arbitrage consistency.

The ACP constructs the closest system, in the selected weighted \(L^1\) metric,

$$
\left(
C_i^{bid,ACP},
C_i^{ask,ACP},
P_i^{bid,ACP},
P_i^{ask,ACP}
\right)_{i=1}^n,
$$

which is bid--ask consistent and contains an internal arbitrage-free certificate.

Therefore, the ACP acts as a cleaning and arbitrage-consistent correction procedure for option quotes before they are used in valuation maps, dynamic hedging, scenario generation, or calibration.


In [ ]:
# ============================================================
# Arbitrage-Consistent Projection directly on bid-ask quotes
# ============================================================
#
# This code implements the ACP procedure directly for bid-ask
# option chains. The output is not a single corrected option
# price. The output is a corrected bid-ask system:
#
#     C_bid_ACP, C_ask_ACP, P_bid_ACP, P_ask_ACP.
#
# The LP also constructs an internal no-arbitrage certificate.
# The certificate is used only to prove consistency of the
# corrected bid-ask system. It is not the main financial output.
#
# ============================================================

import numpy as np
import pandas as pd
from scipy.optimize import linprog
from scipy.stats import norm


# ============================================================
# Black-Scholes prices for testing
# ============================================================

def bs_call_put(S, K, r, sigma, tau):
    """
    Compute Black-Scholes European call and put prices.

    This function is used only for synthetic tests. If option
    prices are imported from a CSV file, an Excel file, a
    database, Bloomberg, Yahoo Finance, or a broker API, then
    this function is not needed.

    Parameters
    ----------
    S : float
        Current underlying price.
    K : array_like
        Strike prices.
    r : float
        Continuously compounded risk-free rate.
    sigma : float
        Volatility.
    tau : float
        Time to maturity.

    Returns
    -------
    call : np.ndarray
        Black-Scholes call prices.
    put : np.ndarray
        Black-Scholes put prices.
    """
    K = np.asarray(K, dtype=float)

    if tau <= 0:
        call = np.maximum(S - K, 0.0)
        put = np.maximum(K - S, 0.0)
        return call, put

    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * tau) / (sigma * np.sqrt(tau))
    d2 = d1 - sigma * np.sqrt(tau)

    call = S * norm.cdf(d1) - K * np.exp(-r * tau) * norm.cdf(d2)
    put = K * np.exp(-r * tau) * norm.cdf(-d2) - S * norm.cdf(-d1)

    return call, put


# ============================================================
# ACP directly on bid-ask quotes
# ============================================================

def acp_direct_bid_ask(
    S,
    K,
    r,
    tau,
    call_bid,
    call_ask,
    put_bid,
    put_ask,
    weights=None,
    lambda_floor=1e-12,
    beta_floor=1e-12,
    return_certificate=True,
):
    """
    Run ACP directly on bid-ask option quotes.

    The routine corrects the bid and ask endpoints directly.

    Input quote vectors:
        call_bid, call_ask, put_bid, put_ask.

    Output quote vectors:
        call_bid_acp, call_ask_acp, put_bid_acp, put_ask_acp.

    The corrected bid-ask system is required to contain an
    internal arbitrage-free frictionless price system. This is
    enforced through a positive dual certificate.

    Mathematical structure
    ----------------------
    Let

        x_0 = 0,
        x_j = K_j, j=1,...,n.

    The certificate variables are

        lambda_0,...,lambda_n,beta > 0.

    They satisfy

        sum_j lambda_j = exp(-r tau),

    and

        sum_j lambda_j x_j + beta = S.

    The internal representative call and put values are

        C_tilde_i = sum_j lambda_j (x_j - K_i)^+ + beta,

        P_tilde_i = sum_j lambda_j (K_i - x_j)^+.

    The corrected bid-ask quotes must satisfy

        C_bid_ACP_i <= C_tilde_i <= C_ask_ACP_i,

        P_bid_ACP_i <= P_tilde_i <= P_ask_ACP_i.

    The final output is the corrected bid-ask system, not the
    internal representative prices.

    Parameters
    ----------
    S : float
        Current underlying price.
    K : array_like
        Strictly increasing strike prices.
    r : float
        Continuously compounded risk-free rate.
    tau : float
        Time to maturity.
    call_bid, call_ask, put_bid, put_ask : array_like
        Raw bid-ask option quotes.
    weights : dict or None
        Optional weights for the weighted L1 projection.
        Expected keys:
            "call_bid", "call_ask", "put_bid", "put_ask".
        If None, all weights are equal to one.
    lambda_floor : float
        Small positive lower bound for lambda variables.
    beta_floor : float
        Small positive lower bound for beta.
    return_certificate : bool
        If True, include the internal certificate in the output.

    Returns
    -------
    result : dict
        Dictionary containing corrected bid-ask quotes and
        diagnostic information.
    """

    # Convert input arrays to NumPy arrays.
    K = np.asarray(K, dtype=float)
    call_bid = np.asarray(call_bid, dtype=float)
    call_ask = np.asarray(call_ask, dtype=float)
    put_bid = np.asarray(put_bid, dtype=float)
    put_ask = np.asarray(put_ask, dtype=float)

    n = len(K)

    # Basic input validation.
    if n == 0:
        raise ValueError("At least one strike is required.")

    if not np.all(np.diff(K) > 0):
        raise ValueError("The strikes K must be strictly increasing.")

    for arr, name in [
        (call_bid, "call_bid"),
        (call_ask, "call_ask"),
        (put_bid, "put_bid"),
        (put_ask, "put_ask"),
    ]:
        if arr.shape != (n,):
            raise ValueError(f"{name} must have shape ({n},).")

    if S <= 0:
        raise ValueError("S must be positive.")

    if tau <= 0:
        raise ValueError("tau must be positive for this ACP implementation.")

    # If no weights are supplied, use equal weights.
    if weights is None:
        weights = {
            "call_bid": np.ones(n),
            "call_ask": np.ones(n),
            "put_bid": np.ones(n),
            "put_ask": np.ones(n),
        }

    w_cb = np.asarray(weights.get("call_bid", np.ones(n)), dtype=float)
    w_ca = np.asarray(weights.get("call_ask", np.ones(n)), dtype=float)
    w_pb = np.asarray(weights.get("put_bid", np.ones(n)), dtype=float)
    w_pa = np.asarray(weights.get("put_ask", np.ones(n)), dtype=float)

    for arr, name in [
        (w_cb, "weights['call_bid']"),
        (w_ca, "weights['call_ask']"),
        (w_pb, "weights['put_bid']"),
        (w_pa, "weights['put_ask']"),
    ]:
        if arr.shape != (n,):
            raise ValueError(f"{name} must have shape ({n},).")
        if np.any(arr < 0):
            raise ValueError(f"{name} must be nonnegative.")

    # Discount factor over the maturity.
    df = np.exp(-r * tau)

    # Nodes used in the finite payoff representation.
    x = np.concatenate(([0.0], K))

    # Payoff matrices.
    #
    # call_payoff[j, i] = (x_j - K_i)^+
    # put_payoff[j, i]  = (K_i - x_j)^+
    call_payoff = np.maximum(x[:, None] - K[None, :], 0.0)
    put_payoff = np.maximum(K[None, :] - x[:, None], 0.0)

    # --------------------------------------------------------
    # Variable indexing
    # --------------------------------------------------------
    #
    # The optimization vector z contains:
    #
    #   lambda_0,...,lambda_n
    #   beta
    #   corrected call bids
    #   corrected call asks
    #   corrected put bids
    #   corrected put asks
    #   absolute deviations for all four quote vectors
    #
    # We use index slices to keep the LP readable.
    # --------------------------------------------------------

    idx_lambda = slice(0, n + 1)
    idx_beta = n + 1

    start = n + 2

    idx_cb = slice(start, start + n)
    start += n

    idx_ca = slice(start, start + n)
    start += n

    idx_pb = slice(start, start + n)
    start += n

    idx_pa = slice(start, start + n)
    start += n

    idx_dcb = slice(start, start + n)
    start += n

    idx_dca = slice(start, start + n)
    start += n

    idx_dpb = slice(start, start + n)
    start += n

    idx_dpa = slice(start, start + n)
    start += n

    num_vars = start

    # --------------------------------------------------------
    # Objective function
    # --------------------------------------------------------
    #
    # We minimize the weighted L1 correction:
    #
    #   sum_i w_cb_i |C_bid_ACP_i - C_bid_i|
    # + sum_i w_ca_i |C_ask_ACP_i - C_ask_i|
    # + sum_i w_pb_i |P_bid_ACP_i - P_bid_i|
    # + sum_i w_pa_i |P_ask_ACP_i - P_ask_i|.
    #
    # The absolute values are linearized through nonnegative
    # deviation variables.
    # --------------------------------------------------------

    c = np.zeros(num_vars)
    c[idx_dcb] = w_cb
    c[idx_dca] = w_ca
    c[idx_dpb] = w_pb
    c[idx_dpa] = w_pa

    # --------------------------------------------------------
    # Equality constraints
    # --------------------------------------------------------
    #
    # These constraints define the discounting and underlying
    # consistency part of the internal dual certificate.
    # --------------------------------------------------------

    A_eq = []
    b_eq = []

    # sum_j lambda_j = exp(-r tau)
    row = np.zeros(num_vars)
    row[idx_lambda] = 1.0
    A_eq.append(row)
    b_eq.append(df)

    # sum_j lambda_j x_j + beta = S
    row = np.zeros(num_vars)
    row[idx_lambda] = x
    row[idx_beta] = 1.0
    A_eq.append(row)
    b_eq.append(S)

    # --------------------------------------------------------
    # Inequality constraints
    # --------------------------------------------------------

    A_ub = []
    b_ub = []

    # 1. Bid-ask consistency:
    #
    #     C_bid_ACP_i <= C_ask_ACP_i
    #     P_bid_ACP_i <= P_ask_ACP_i
    #
    for i in range(n):
        row = np.zeros(num_vars)
        row[idx_cb.start + i] = 1.0
        row[idx_ca.start + i] = -1.0
        A_ub.append(row)
        b_ub.append(0.0)

        row = np.zeros(num_vars)
        row[idx_pb.start + i] = 1.0
        row[idx_pa.start + i] = -1.0
        A_ub.append(row)
        b_ub.append(0.0)

    # 2. Internal certificate inside corrected bid-ask intervals.
    #
    # For each call:
    #
    #     C_bid_ACP_i <= C_tilde_i <= C_ask_ACP_i.
    #
    # For each put:
    #
    #     P_bid_ACP_i <= P_tilde_i <= P_ask_ACP_i.
    #
    # These are written as linear inequalities.
    #
    for i in range(n):

        # C_bid_ACP_i <= C_tilde_i
        # C_bid_ACP_i - payoff_i^T lambda - beta <= 0
        row = np.zeros(num_vars)
        row[idx_cb.start + i] = 1.0
        row[idx_lambda] = -call_payoff[:, i]
        row[idx_beta] = -1.0
        A_ub.append(row)
        b_ub.append(0.0)

        # C_tilde_i <= C_ask_ACP_i
        # payoff_i^T lambda + beta - C_ask_ACP_i <= 0
        row = np.zeros(num_vars)
        row[idx_lambda] = call_payoff[:, i]
        row[idx_beta] = 1.0
        row[idx_ca.start + i] = -1.0
        A_ub.append(row)
        b_ub.append(0.0)

        # P_bid_ACP_i <= P_tilde_i
        # P_bid_ACP_i - payoff_i^T lambda <= 0
        row = np.zeros(num_vars)
        row[idx_pb.start + i] = 1.0
        row[idx_lambda] = -put_payoff[:, i]
        A_ub.append(row)
        b_ub.append(0.0)

        # P_tilde_i <= P_ask_ACP_i
        # payoff_i^T lambda - P_ask_ACP_i <= 0
        row = np.zeros(num_vars)
        row[idx_lambda] = put_payoff[:, i]
        row[idx_pa.start + i] = -1.0
        A_ub.append(row)
        b_ub.append(0.0)

    # 3. Absolute deviation constraints.
    #
    # For each corrected endpoint y and raw endpoint y0, introduce
    # d >= 0 and impose:
    #
    #     y - y0 <= d,
    #     y0 - y <= d.
    #
    # Equivalently:
    #
    #     y - d <= y0,
    #    -y - d <= -y0.
    #
    def add_abs_constraints(price_slice, dev_slice, raw):
        for i in range(n):

            # corrected - deviation <= raw
            row = np.zeros(num_vars)
            row[price_slice.start + i] = 1.0
            row[dev_slice.start + i] = -1.0
            A_ub.append(row)
            b_ub.append(raw[i])

            # -corrected - deviation <= -raw
            row = np.zeros(num_vars)
            row[price_slice.start + i] = -1.0
            row[dev_slice.start + i] = -1.0
            A_ub.append(row)
            b_ub.append(-raw[i])

    add_abs_constraints(idx_cb, idx_dcb, call_bid)
    add_abs_constraints(idx_ca, idx_dca, call_ask)
    add_abs_constraints(idx_pb, idx_dpb, put_bid)
    add_abs_constraints(idx_pa, idx_dpa, put_ask)

    # --------------------------------------------------------
    # Variable bounds
    # --------------------------------------------------------

    bounds = []

    # Positive lower bounds for lambda variables.
    for _ in range(n + 1):
        bounds.append((lambda_floor, None))

    # Positive lower bound for beta.
    bounds.append((beta_floor, None))

    # Corrected quotes are nonnegative.
    for _ in range(4 * n):
        bounds.append((0.0, None))

    # Deviation variables are nonnegative.
    for _ in range(4 * n):
        bounds.append((0.0, None))

    # --------------------------------------------------------
    # Solve the linear programme
    # --------------------------------------------------------

    res = linprog(
        c,
        A_ub=np.asarray(A_ub),
        b_ub=np.asarray(b_ub),
        A_eq=np.asarray(A_eq),
        b_eq=np.asarray(b_eq),
        bounds=bounds,
        method="highs",
    )

    if not res.success:
        raise RuntimeError(f"ACP bid-ask LP failed: {res.message}")

    z = res.x

    # Main ACP outputs: corrected bid-ask endpoints.
    result = {
        "call_bid_acp": z[idx_cb],
        "call_ask_acp": z[idx_ca],
        "put_bid_acp": z[idx_pb],
        "put_ask_acp": z[idx_pa],
        "objective": res.fun,
        "success": res.success,
        "message": res.message,
    }

    # Optional diagnostic certificate.
    if return_certificate:
        lam = z[idx_lambda]
        beta = z[idx_beta]

        certificate_call = call_payoff.T @ lam + beta
        certificate_put = put_payoff.T @ lam

        result["lambda"] = lam
        result["beta"] = beta
        result["certificate_call"] = certificate_call
        result["certificate_put"] = certificate_put

    return result


# ============================================================
# Diagnostic checks
# ============================================================

def check_acp_bid_ask_result(K, result, tol=1e-8):
    """
    Print diagnostic checks for an ACP bid-ask result.

    This routine verifies:
        1. call bid <= call ask,
        2. put bid <= put ask,
        3. nonnegativity of corrected quotes,
        4. if returned, the internal certificate lies inside
           the corrected bid-ask intervals.
    """
    K = np.asarray(K, dtype=float)

    cb = result["call_bid_acp"]
    ca = result["call_ask_acp"]
    pb = result["put_bid_acp"]
    pa = result["put_ask_acp"]

    print("ACP status:", result["message"])
    print("ACP objective:", result["objective"])
    print()
    print("call bid <= call ask:", np.all(cb <= ca + tol))
    print("put bid <= put ask:", np.all(pb <= pa + tol))
    print("nonnegative call bid:", np.all(cb >= -tol))
    print("nonnegative call ask:", np.all(ca >= -tol))
    print("nonnegative put bid:", np.all(pb >= -tol))
    print("nonnegative put ask:", np.all(pa >= -tol))

    if "certificate_call" in result and "certificate_put" in result:
        cc = result["certificate_call"]
        pc = result["certificate_put"]

        print()
        print(
            "internal call certificate inside corrected bid-ask:",
            np.all((cb <= cc + tol) & (cc <= ca + tol)),
        )
        print(
            "internal put certificate inside corrected bid-ask:",
            np.all((pb <= pc + tol) & (pc <= pa + tol)),
        )


def build_result_table(K, call_bid, call_ask, put_bid, put_ask, result, call_true=None, put_true=None):
    """
    Build a pandas table comparing raw quotes and ACP-corrected quotes.

    Parameters
    ----------
    K : array_like
        Strikes.
    call_bid, call_ask, put_bid, put_ask : array_like
        Raw bid-ask quotes.
    result : dict
        Output of acp_direct_bid_ask.
    call_true, put_true : array_like or None
        Optional benchmark prices, for example Black-Scholes prices.

    Returns
    -------
    table : pandas.DataFrame
    """
    table = pd.DataFrame({
        "K": K,
        "C_bid_raw": call_bid,
        "C_ask_raw": call_ask,
        "C_bid_ACP": result["call_bid_acp"],
        "C_ask_ACP": result["call_ask_acp"],
        "P_bid_raw": put_bid,
        "P_ask_raw": put_ask,
        "P_bid_ACP": result["put_bid_acp"],
        "P_ask_ACP": result["put_ask_acp"],
    })

    if call_true is not None:
        table.insert(1, "C_BS", call_true)

    if put_true is not None:
        # Place P_BS before put raw quotes.
        loc = list(table.columns).index("P_bid_raw")
        table.insert(loc, "P_BS", put_true)

    return table


# ============================================================
# Synthetic test data
# ============================================================

def make_disturbed_bid_ask_test(
    S=100.0,
    r=0.03,
    sigma=0.20,
    tau=0.50,
    seed=123,
):
    """
    Generate synthetic bid-ask option quotes from Black-Scholes and
    deliberately perturb them.

    This function is only for testing. In real applications, replace
    this with data imported from a CSV file, an Excel file, a database,
    or an API.
    """
    rng = np.random.default_rng(seed)

    # Example strike grid.
    K = np.arange(70.0, 131.0, 5.0)

    # Frictionless benchmark Black-Scholes prices.
    call_true, put_true = bs_call_put(S, K, r, sigma, tau)

    # Create base bid-ask spreads.
    call_spread = 0.04 * np.maximum(call_true, 1.0) + 0.05
    put_spread = 0.04 * np.maximum(put_true, 1.0) + 0.05

    # Add random noise to mid prices.
    call_mid = call_true + rng.normal(0.0, 0.35, len(K))
    put_mid = put_true + rng.normal(0.0, 0.35, len(K))

    call_bid = call_mid - 0.5 * call_spread
    call_ask = call_mid + 0.5 * call_spread
    put_bid = put_mid - 0.5 * put_spread
    put_ask = put_mid + 0.5 * put_spread

    # Enforce nonnegative raw bids before injecting violations.
    call_bid = np.maximum(call_bid, 0.0)
    put_bid = np.maximum(put_bid, 0.0)

    # Deliberate crossed-market and static-consistency violations.
    if len(K) >= 9:
        # Crossed call quote.
        call_bid[4] = call_ask[4] + 0.60

        # Crossed put quote.
        put_bid[8] = put_ask[8] + 0.50

        # Additional local distortions.
        call_bid[6] += 1.50
        call_ask[7] = max(call_ask[7] - 1.25, 0.0)

        put_bid[5] += 1.20
        put_ask[6] = max(put_ask[6] - 1.00, 0.0)

    return {
        "S": S,
        "r": r,
        "sigma": sigma,
        "tau": tau,
        "K": K,
        "call_true": call_true,
        "put_true": put_true,
        "call_bid": call_bid,
        "call_ask": call_ask,
        "put_bid": put_bid,
        "put_ask": put_ask,
    }


# ============================================================
# Optional helper: weights based on bid-ask spreads
# ============================================================

def inverse_spread_weights(call_bid, call_ask, put_bid, put_ask, eps=1e-6, cap=1e3):
    """
    Construct inverse-spread weights.

    Tight spreads receive larger weights and are therefore moved
    less by the ACP. Wide spreads receive smaller weights and are
    allowed to adjust more.

    If raw quotes are crossed, the spread is clipped below by eps.
    """
    call_bid = np.asarray(call_bid, dtype=float)
    call_ask = np.asarray(call_ask, dtype=float)
    put_bid = np.asarray(put_bid, dtype=float)
    put_ask = np.asarray(put_ask, dtype=float)

    call_spread = np.maximum(call_ask - call_bid, eps)
    put_spread = np.maximum(put_ask - put_bid, eps)

    w_call = np.clip(1.0 / call_spread, 0.0, cap)
    w_put = np.clip(1.0 / put_spread, 0.0, cap)

    return {
        "call_bid": w_call,
        "call_ask": w_call,
        "put_bid": w_put,
        "put_ask": w_put,
    }


# ============================================================
# Example 1: synthetic Black-Scholes test with disturbed bid-ask
# ============================================================

data = make_disturbed_bid_ask_test()

S = data["S"]
r = data["r"]
tau = data["tau"]
K = data["K"]

call_bid = data["call_bid"]
call_ask = data["call_ask"]
put_bid = data["put_bid"]
put_ask = data["put_ask"]

# Use equal weights by default.
# To use inverse-spread weights, replace weights=None by:
#
#     weights=inverse_spread_weights(call_bid, call_ask, put_bid, put_ask)
#
result = acp_direct_bid_ask(
    S=S,
    K=K,
    r=r,
    tau=tau,
    call_bid=call_bid,
    call_ask=call_ask,
    put_bid=put_bid,
    put_ask=put_ask,
    weights=None,
    lambda_floor=1e-12,
    beta_floor=1e-12,
    return_certificate=True,
)

check_acp_bid_ask_result(K, result)

table = build_result_table(
    K=K,
    call_bid=call_bid,
    call_ask=call_ask,
    put_bid=put_bid,
    put_ask=put_ask,
    result=result,
    call_true=data["call_true"],
    put_true=data["put_true"],
)

display(table.round(6))


# ============================================================
# Example 2: template for real data from CSV
# ============================================================
#
# Uncomment and adapt this block when you have a CSV file.
#
# Expected columns:
#
#     strike, call_bid, call_ask, put_bid, put_ask
#
# Optional columns:
#
#     tau, maturity
#
# ------------------------------------------------------------
#
# df = pd.read_csv("option_chain.csv")
# df = df.sort_values("strike")
#
# K = df["strike"].to_numpy(float)
# call_bid = df["call_bid"].to_numpy(float)
# call_ask = df["call_ask"].to_numpy(float)
# put_bid = df["put_bid"].to_numpy(float)
# put_ask = df["put_ask"].to_numpy(float)
#
# S = 100.0      # Replace by the current underlying price.
# r = 0.03       # Replace by the relevant continuously compounded rate.
# tau = 0.50     # Replace by the time to maturity in years.
#
# result = acp_direct_bid_ask(
#     S=S,
#     K=K,
#     r=r,
#     tau=tau,
#     call_bid=call_bid,
#     call_ask=call_ask,
#     put_bid=put_bid,
#     put_ask=put_ask,
#     weights=None,
#     return_certificate=True,
# )
#
# check_acp_bid_ask_result(K, result)
#
# df["call_bid_acp"] = result["call_bid_acp"]
# df["call_ask_acp"] = result["call_ask_acp"]
# df["put_bid_acp"] = result["put_bid_acp"]
# df["put_ask_acp"] = result["put_ask_acp"]
#
# display(df)


# ============================================================
# Example 3: template for several maturities in one DataFrame
# ============================================================
#
# The ACP should be applied separately for each maturity.
#
# Expected columns:
#
#     maturity, tau, strike, call_bid, call_ask, put_bid, put_ask
#
# ------------------------------------------------------------
#
# results = []
#
# for maturity, g in df.groupby("maturity"):
#     g = g.sort_values("strike").copy()
#
#     K = g["strike"].to_numpy(float)
#     call_bid = g["call_bid"].to_numpy(float)
#     call_ask = g["call_ask"].to_numpy(float)
#     put_bid = g["put_bid"].to_numpy(float)
#     put_ask = g["put_ask"].to_numpy(float)
#
#     tau = float(g["tau"].iloc[0])
#
#     res = acp_direct_bid_ask(
#         S=S,
#         K=K,
#         r=r,
#         tau=tau,
#         call_bid=call_bid,
#         call_ask=call_ask,
#         put_bid=put_bid,
#         put_ask=put_ask,
#         weights=None,
#         return_certificate=True,
#     )
#
#     g["call_bid_acp"] = res["call_bid_acp"]
#     g["call_ask_acp"] = res["call_ask_acp"]
#     g["put_bid_acp"] = res["put_bid_acp"]
#     g["put_ask_acp"] = res["put_ask_acp"]
#
#     results.append(g)
#
# df_acp = pd.concat(results, ignore_index=True)
# display(df_acp)
